# Calidad y conciliaciones

Controles sobre ventas, inventario y archivo manual de escenario. Los resultados se reproducen desde las fuentes.

In [1]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'src' / 'pipeline.py').is_file(), 'Abre Jupyter desde la raíz o notebooks/'


## 1. Resumen de controles

In [2]:
import sys
sys.path.insert(0,str(ROOT/'src'))
from pipeline import load,analyse_sales,interval_quality
from analitica import calendario_ventas,cobertura_temporal,calidad_inventario,homologacion_muestra,clasificacion_xyz,clasificacion_abc
sales,stock=load()
analyse_sales(sales)


,control,value,unit
0,filas_ventas,125751,registros
1,duplicados_clave_ventas,0,registros
2,devoluciones_filas,7491,registros
3,devoluciones_cantidad_no_negativa,0,registros
4,devoluciones_venta_no_negativa,0,registros
5,devoluciones_cogs_no_negativo,0,registros
6,razon_devolucion_filas_no_nulas,0,registros
7,ventas_brutas,11215681.26,importe
8,importe_devoluciones,-733571.01,importe
9,ventas_netas,10482110.25,importe


## 2. Diferencias de valorización

In [3]:
calidad=calidad_inventario(stock)
excepciones=calidad.loc[calidad['valorizacion_fuera_tolerancia']|calidad['existencia_negativa']]
excepciones.head(12)


,Store,Product No,Start Date,End Date,Qty on hand,Cost of Stocks,Stock Unit Cost Price,Supplier,valor_calculado,diferencia_valorizacion,valorizacion_fuera_tolerancia,existencia_negativa,existencia_cero_valorizada
5,STR-1163,PROD-124597,2025-06-09,2026-01-07,0,-6.80,0.00,Vendor 0166,0.00,-6.80,True,False,True
11,STR-1145,PROD-100353,2025-09-06,2026-01-05,-1,-59.17,59.17,Vendor 0166,-59.17,0.00,False,True,False
12,STR-1305,PROD-154191,2025-08-27,2026-01-05,0,5.46,0.00,Vendor 0010,0.00,5.46,True,False,True
13,STR-1010,PROD-138976,2025-12-10,2026-01-05,0,-0.15,0.00,Vendor 0166,0.00,-0.15,True,False,True
14,STR-1075,PROD-165960,2025-09-04,2026-01-05,0,-1.63,0.00,Vendor 0169,0.00,-1.63,True,False,True
145,STR-1006,PROD-139867,2025-10-20,2025-11-13,0,0.11,0.00,Vendor 0169,0.00,0.11,True,False,True
148,STR-1107,PROD-139862,2025-09-08,2025-11-13,0,0.17,0.00,Vendor 0169,0.00,0.17,True,False,True
207,STR-1145,PROD-139829,2025-10-13,2025-10-28,0,-1.40,0.00,Vendor 0134,0.00,-1.40,True,False,True
236,STR-1094,PROD-165963,2025-08-07,2025-10-05,0,3.74,0.00,Vendor 0169,0.00,3.74,True,False,True
241,STR-1074,PROD-193905,2025-09-24,2025-09-25,0,0.48,0.00,Vendor 0166,0.00,0.48,True,False,True


In [4]:
excepciones[['diferencia_valorizacion']].agg(['sum','min','max'])


,diferencia_valorizacion
sum,-580.1494
min,-139.0400
max,245.4700


## 3. Disponibilidad de información

In [5]:
cruces=cobertura_temporal(calendario_ventas(sales),stock)
cruces.groupby(['estado_cobertura','causa_sin_dato']).agg(filas=('id_venta','size'),ventas=('Sales Amount','sum'))


filas      ventas
estado_cobertura causa_sin_dato                             
SIN_DATO         ANTES_PRIMER_INTERVALO     1066    66347.00
                 DESPUES_ULTIMO_INTERVALO  22898  2254318.33
                 ENTRE_INTERVALOS           6479   658346.09
                 PAR_SIN_INVENTARIO         2717   211340.65
STOCK_CERO       CON_INTERVALO             10581  1083294.56
STOCK_NEGATIVO   CON_INTERVALO               878    66661.34
STOCK_POSITIVO   CON_INTERVALO             81132  6141802.28

## 4. Homologación controlada de catálogos

In [6]:
ventas=calendario_ventas(sales)
productos=clasificacion_xyz(ventas,clasificacion_abc(ventas))
homologacion=homologacion_muestra(productos,ventas)
homologacion['estado_homologacion'].value_counts()


estado_homologacion
CORRECTO               2298
CODIGO_AMBIGUO           16
SIN_CORRESPONDENCIA      12
Name: count, dtype: int64

Las incidencias del catálogo son casos de prueba generados con reglas deterministas.

## 5. Ingesta manual y decisiones de aceptación

In [7]:
from fuentes_manuales import validar_ajustes
ajustes=validar_ajustes(ROOT / "input/manual/Ajustes_Comerciales_escenario.csv", set(sales["Product No"]), set(sales["Store"]))
display(ajustes[["documento","estado","resultado_validacion","incidencias"]])

,documento,estado,resultado_validacion,incidencias
0,AJ-001,APROBADO,VALIDO,
1,AJ-002,APROBADO,VALIDO,
2,AJ-003,PENDIENTE,VALIDO,
3,AJ-004,APROBADO,RECHAZADO,TIENDA_DESCONOCIDA
4,AJ-005,APROBADO,RECHAZADO,PRODUCTO_DESCONOCIDO
5,AJ-006,APROBADO,RECHAZADO,DOCUMENTO_DUPLICADO
6,AJ-006,APROBADO,RECHAZADO,DOCUMENTO_DUPLICADO
7,AJ-007,APROBADO,RECHAZADO,IMPORTE_INVALIDO
8,AJ-008,APROBADO,VALIDO,


## 6. Comprobaciones automáticas

In [8]:
import sys
sys.path.insert(0, str(ROOT / 'src'))
from validaciones import comprobar
comprobar()

{'controles': 48, 'estado': 'OK'}